# Persist daily WBGT, Tg, and Tnwb pairwise statistics

Each date is computed in a fresh child process because reusing a process caused progressive engine slowdowns during long runs; limiting execution to one child at a time also bounds memory use. 

WARNING: this also allows concurrent execution across multiple dates, implemented in `scripts/daily_pairwise.py`; only do so if free system RAM is sufficient to hold concurrent child processes.

The notebook kernel publishes every Parquet partition synchronously to keep output replacement deterministic. 

Each execution computes all three quantities and independent coverage from one wide materialized base relation. Agreement requires joint validity across WBGT, Tg, and Tnwb. Date-scoped atomic directory replacement makes interrupted runs safe to resume and completed dates safe to rerun.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

from liljegren_analysis import (
    HSBRunResults, compute_daily_pairwise, initialize_pairwise_output,
    persist_daily_pairwise,
)


In [ ]:
working_directory = Path.cwd().resolve()
repository = next((candidate for candidate in (working_directory, *working_directory.parents) if (candidate / "pyproject.toml").is_file()), None)
if repository is None:
    raise FileNotFoundError("Could not locate the repository root")

runs = (repository.parent / "HeatStressBench" / ".hsb" / "runs" / "wbgt-comparison-20260902-150355",)
runs = tuple(path.resolve() for path in runs)
output_root = repository / "data" / "pairwise"
sample_scope = "all-implementations"
replace_existing_output = True
start_date = date(2025, 1, 1)
end_date = date(2025, 12, 31)

if start_date > end_date:
    raise ValueError("start_date must not be after end_date")
dates = [start_date + timedelta(days=offset) for offset in range((end_date - start_date).days + 1)]
len(dates)


## Process and persist

The executor has one live child and replaces it after every task. Daily atomic partition replacement makes reruns resumable and idempotent.

In [ ]:
for run in runs:
    HSBRunResults.from_hsb_run(run)
initialize_pairwise_output(
    output_root, sample_scope, overwrite=replace_existing_output
)
executor = ProcessPoolExecutor(max_workers=1, max_tasks_per_child=1)
try:
    for completed, result_date in enumerate(dates, start=1):
        processed_date, summaries, worker_pid, compute_seconds = executor.submit(
            compute_daily_pairwise, result_date, runs, sample_scope
        ).result()
        if processed_date != result_date:
            raise RuntimeError(f"Expected {result_date}, received {processed_date}")
        persist_daily_pairwise(output_root, result_date, summaries, sample_scope)
        processed_at = datetime.now(ZoneInfo("Asia/Singapore"))
        print(f"[{processed_at:%Y-%m-%d %H:%M:%S SGT}] Processed {result_date} worker_pid={worker_pid} compute={compute_seconds:.1f}s [{completed}/{len(dates)}]")
finally:
    executor.shutdown(wait=True, cancel_futures=True)

print(f"Saved pairwise partitions to {output_root}")
